In [37]:
import glob
import os
import re
import pandas as pd
import mdtraj as md
from prody import parsePDB, parseDCD, calcRMSD, calcRMSF, EDA
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib
import natsort
from prody import calcSubspaceOverlap

In [29]:
base_gen_dir = "../Model_results"
gen_files = glob.glob(
    os.path.join(base_gen_dir, "*", "*", "*", "fraction_*", "generated_filtered_*.xtc")
)
rows = []
for fl in gen_files:
    norm_fl = os.path.normpath(fl)
    parts = norm_fl.split(os.sep)
    try:
        filename   = parts[-1]
        frac_folder = parts[-2]
        rep_folder  = parts[-3]
        pdb_code    = parts[-4]
        condition   = parts[-5]
        rep_match = re.match(rf"{re.escape(pdb_code)}_rep_(\d+)$", rep_folder)
        replica = int(rep_match.group(1))
        frac_match = re.match(r"fraction_(\d+)$", frac_folder)
        fraction = int(frac_match.group(1))
        file_frac_match = re.match(r"generated_filtered_(\d+)\.xtc$", filename)
        pdb_file = os.path.join("..", "data", "palantir_data", pdb_code, "backbone.pdb")

        rows.append({
            "source": "generated",
            "condition": condition,
            "pdb_code": pdb_code,
            "replica": replica,
            "fraction": fraction,
            "xtc_path": norm_fl,
            "dcd_path": None,
            "pdb_file": pdb_file
        })
    except Exception as e:
        print(f"[WARNING] file: {fl} | error: {e}")
df_gen = pd.DataFrame(rows)
if not df_gen.empty:
    df_gen = df_gen.sort_values(
        by=["condition", "pdb_code", "replica", "fraction"]
    ).reset_index(drop=True)

display(df_gen)

,source,condition,pdb_code,replica,fraction,xtc_path,dcd_path,pdb_file
0,generated,not_shuffle_allframes,3UTQ,0,10,../Model_results/not_shuffle_allframes/3UTQ/3U...,None,../data/palantir_data/3UTQ/backbone.pdb
1,generated,not_shuffle_allframes,3UTQ,0,20,../Model_results/not_shuffle_allframes/3UTQ/3U...,None,../data/palantir_data/3UTQ/backbone.pdb
2,generated,not_shuffle_allframes,3UTQ,0,30,../Model_results/not_shuffle_allframes/3UTQ/3U...,None,../data/palantir_data/3UTQ/backbone.pdb
3,generated,not_shuffle_allframes,3UTQ,0,40,../Model_results/not_shuffle_allframes/3UTQ/3U...,None,../data/palantir_data/3UTQ/backbone.pdb
4,generated,not_shuffle_allframes,3UTQ,0,50,../Model_results/not_shuffle_allframes/3UTQ/3U...,None,../data/palantir_data/3UTQ/backbone.pdb
...,...,...,...,...,...,...,...,...
331,generated,shuffle_not_allframes,pep_free,2,30,../Model_results/shuffle_not_allframes/pep_fre...,None,../data/palantir_data/pep_free/backbone.pdb
332,generated,shuffle_not_allframes,pep_free,2,40,../Model_results/shuffle_not_allframes/pep_fre...,None,../data/palantir_data/pep_free/backbone.pdb
333,generated,shuffle_not_allframes,pep_free,2,50,../Model_results/shuffle_not_allframes/pep_fre...,None,../data/palantir_data/pep_free/backbone.pdb
334,generated,shuffle_not_allframes,pep_free,2,60,../Model_results/shuffle_not_allframes/pep_fre...,None,../data/palantir_data/pep_free/backbone.pdb


In [30]:
base_md_dir = "../data/palantir_data" 
md_files = glob.glob(
    os.path.join(base_md_dir, "*", "*_rep_*_backbone.xtc")
)
rows = []
for fl in md_files:
    norm_fl = os.path.normpath(fl)
    parts = norm_fl.split(os.sep)
    try:
        pdb_code = parts[-2]
        filename = parts[-1]
        rep_match = re.match(rf"{re.escape(pdb_code)}_rep_(\d+)_backbone\.xtc", filename)
        replica = int(rep_match.group(1))
        pdb_file = os.path.join(base_md_dir, pdb_code, "backbone.pdb")
        rows.append({
            "source": "md",
            "pdb_code": pdb_code,
            "replica": replica,
            "xtc_path": norm_fl,
            "dcd_path": None,  
            "pdb_file": pdb_file
        })
    except Exception as e:
        print(f"[WARNING] file: {fl} | error: {e}")
df_md = pd.DataFrame(rows)
if not df_md.empty:
    df_md = df_md.sort_values(
        by=["pdb_code", "replica"]
    ).reset_index(drop=True)
display(df_md)

,source,pdb_code,replica,xtc_path,dcd_path,pdb_file
0,md,3UTQ,0,../data/palantir_data/3UTQ/3UTQ_rep_0_backbone...,None,../data/palantir_data/3UTQ/backbone.pdb
1,md,3UTQ,1,../data/palantir_data/3UTQ/3UTQ_rep_1_backbone...,None,../data/palantir_data/3UTQ/backbone.pdb
2,md,3UTQ,2,../data/palantir_data/3UTQ/3UTQ_rep_2_backbone...,None,../data/palantir_data/3UTQ/backbone.pdb
3,md,5C0F,0,../data/palantir_data/5C0F/5C0F_rep_0_backbone...,None,../data/palantir_data/5C0F/backbone.pdb
4,md,5C0F,1,../data/palantir_data/5C0F/5C0F_rep_1_backbone...,None,../data/palantir_data/5C0F/backbone.pdb
5,md,5C0F,2,../data/palantir_data/5C0F/5C0F_rep_2_backbone...,None,../data/palantir_data/5C0F/backbone.pdb
6,md,5N1Y,0,../data/palantir_data/5N1Y/5N1Y_rep_0_backbone...,None,../data/palantir_data/5N1Y/backbone.pdb
7,md,5N1Y,1,../data/palantir_data/5N1Y/5N1Y_rep_1_backbone...,None,../data/palantir_data/5N1Y/backbone.pdb
8,md,5N1Y,2,../data/palantir_data/5N1Y/5N1Y_rep_2_backbone...,None,../data/palantir_data/5N1Y/backbone.pdb
9,md,pep_free,0,../data/palantir_data/pep_free/pep_free_rep_0_...,None,../data/palantir_data/pep_free/backbone.pdb


In [31]:
dcd_root = "Converted_DCD"
os.makedirs(dcd_root, exist_ok=True)

for i, row in df_md.iterrows():
    pdb_code = row["pdb_code"]
    replica = row["replica"]
    xtc_path = row["xtc_path"]
    pdb_file = row["pdb_file"]

    out_dir = os.path.join(dcd_root, "md", pdb_code)
    os.makedirs(out_dir, exist_ok=True)

    dcd_path = os.path.join(out_dir, f"{pdb_code}_rep_{replica}_backbone.dcd")

    try:
        print(f"[INFO] Converting MD: {pdb_code} | rep {replica}")

        traj = md.load_xtc(xtc_path, top=pdb_file)
        traj.save_dcd(dcd_path)

        df_md.at[i, "dcd_path"] = dcd_path

    except Exception as e:
        print(f"[ERROR] Failed MD conversion: {pdb_code} | rep {replica} | {e}")

for i, row in df_gen.iterrows():
    pdb_code = row["pdb_code"]
    replica = row["replica"]
    condition = row["condition"]
    fraction = row["fraction"]
    xtc_path = row["xtc_path"]
    pdb_file = row["pdb_file"]

    out_dir = os.path.join(
        dcd_root,
        "generated",
        condition,
        pdb_code,
        f"{pdb_code}_rep_{replica}",
        f"fraction_{fraction}"
    )
    os.makedirs(out_dir, exist_ok=True)

    dcd_path = os.path.join(out_dir, f"generated_filtered_{fraction}.dcd")

    try:
        print(f"[INFO] Converting generated: {condition} | {pdb_code} | rep {replica} | frac {fraction}")

        traj = md.load_xtc(xtc_path, top=pdb_file)
        traj.save_dcd(dcd_path)

        df_gen.at[i, "dcd_path"] = dcd_path

    except Exception as e:
        print(f"[ERROR] Failed generated conversion: {condition} | {pdb_code} | rep {replica} | frac {fraction} | {e}")

metadata_dir = "Analysis_results/metadata"
os.makedirs(metadata_dir, exist_ok=True)

df_md.to_csv(os.path.join(metadata_dir, "df_md.csv"), index=False)
df_gen.to_csv(os.path.join(metadata_dir, "df_gen.csv"), index=False)

print("\n[INFO] DCD conversion finished.")
print(f"[INFO] MD converted: {df_md['dcd_path'].notna().sum()} / {len(df_md)}")
print(f"[INFO] Generated converted: {df_gen['dcd_path'].notna().sum()} / {len(df_gen)}")

[INFO] Converting MD: 3UTQ | rep 0
[INFO] Converting MD: 3UTQ | rep 1
[INFO] Converting MD: 3UTQ | rep 2
[INFO] Converting MD: 5C0F | rep 0
[INFO] Converting MD: 5C0F | rep 1
[INFO] Converting MD: 5C0F | rep 2
[INFO] Converting MD: 5N1Y | rep 0
[INFO] Converting MD: 5N1Y | rep 1
[INFO] Converting MD: 5N1Y | rep 2
[INFO] Converting MD: pep_free | rep 0
[INFO] Converting MD: pep_free | rep 1
[INFO] Converting MD: pep_free | rep 2
[INFO] Converting generated: not_shuffle_allframes | 3UTQ | rep 0 | frac 10
[INFO] Converting generated: not_shuffle_allframes | 3UTQ | rep 0 | frac 20
[INFO] Converting generated: not_shuffle_allframes | 3UTQ | rep 0 | frac 30
[INFO] Converting generated: not_shuffle_allframes | 3UTQ | rep 0 | frac 40
[INFO] Converting generated: not_shuffle_allframes | 3UTQ | rep 0 | frac 50
[INFO] Converting generated: not_shuffle_allframes | 3UTQ | rep 0 | frac 60
[INFO] Converting generated: not_shuffle_allframes | 3UTQ | rep 0 | frac 70
[INFO] Converting generated: not_shu

In [32]:
print(df_md[["pdb_code", "replica", "dcd_path"]].head())
print(df_gen[["condition", "pdb_code", "replica", "fraction", "dcd_path"]].head())

  pdb_code  replica                                       dcd_path
0     3UTQ        0  Converted_DCD/md/3UTQ/3UTQ_rep_0_backbone.dcd
1     3UTQ        1  Converted_DCD/md/3UTQ/3UTQ_rep_1_backbone.dcd
2     3UTQ        2  Converted_DCD/md/3UTQ/3UTQ_rep_2_backbone.dcd
3     5C0F        0  Converted_DCD/md/5C0F/5C0F_rep_0_backbone.dcd
4     5C0F        1  Converted_DCD/md/5C0F/5C0F_rep_1_backbone.dcd
               condition pdb_code  replica  fraction  \
0  not_shuffle_allframes     3UTQ        0        10   
1  not_shuffle_allframes     3UTQ        0        20   
2  not_shuffle_allframes     3UTQ        0        30   
3  not_shuffle_allframes     3UTQ        0        40   
4  not_shuffle_allframes     3UTQ        0        50   

                                            dcd_path  
0  Converted_DCD/generated/not_shuffle_allframes/...  
1  Converted_DCD/generated/not_shuffle_allframes/...  
2  Converted_DCD/generated/not_shuffle_allframes/...  
3  Converted_DCD/generated/not_shuffle_al

In [ ]:
out_dir = "Analysis_results/trajectory_metrics"
os.makedirs(out_dir, exist_ok=True)

df_all = pd.concat([df_md, df_gen], ignore_index=True)
rmsd_rows = []
rmsf_rows = []
rmsf_pep_rows = []

for i, row in df_all.iterrows():
    try:
        source = row.get("source")
        condition = row.get("condition", None)
        fraction = row.get("fraction", None)
        pdb_code = row["pdb_code"]
        replica = row["replica"]

        traj_path = row["dcd_path"] if pd.notna(row["dcd_path"]) else row["xtc_path"]
        pdb_file = row["pdb_file"]

        print(f"[INFO] {i+1}/{len(df_all)} | {source} | {pdb_code} | rep {replica}")

        # load
        traj = parseDCD(traj_path)
        syst = parsePDB(pdb_file)
        traj.setAtoms(syst)

        # ---------------- RMSD ----------------
        ca = syst.select("name CA")
        traj.setAtoms(ca)
        traj.superpose()
        for f in range(1, len(traj)):
            val = calcRMSD(traj[0], traj[f])
            rmsd_rows.append({
                "source": source,
                "condition": condition,
                "fraction": fraction,
                "pdb_code": pdb_code,
                "replica": replica,
                "frame": f,
                "rmsd": float(val)
            })
        # ---------------- RMSF peptide ----------------
        if pdb_code != "pep_free":
            align = syst.select("name CA and chain A and resid 1 to 180")
            pep = syst.select("name CA and chain C")

            if align is not None and pep is not None:
                traj.setAtoms(align)
                traj.superpose()

                traj.setAtoms(pep)
                rmsf_pep = calcRMSF(traj)

                for j, val in enumerate(rmsf_pep):
                    rmsf_pep_rows.append({
                        "source": source,
                        "condition": condition,
                        "fraction": fraction,
                        "pdb_code": pdb_code,
                        "replica": replica,
                        "pep_pos": f"P{j+1}",
                        "rmsf_pep": float(val)
                    })
            else:
                print(f"[WARNING] Missing peptide selection: {pdb_code} rep {replica}")
        # ---------------- RMSF overall ----------------
        sel = syst.select(
            "name CA and chain A and resid 1 to 274 or "
            "name CA and chain B and resid 1 to 99 or "
            "name CA and chain C"
        )
        if sel is None:
            print(f"[WARNING] RMSF selection empty: {pdb_code} rep {replica}")
            continue

        traj.setAtoms(sel)
        traj.superpose()

        rmsf = calcRMSF(traj)
        resi = sel.getResindices()

        for r, val in zip(resi, rmsf):
            rmsf_rows.append({
                "source": source,
                "condition": condition,
                "fraction": fraction,
                "pdb_code": pdb_code,
                "replica": replica,
                "resindex": int(r),
                "rmsf": float(val)
            })
    except Exception as e:
        print(f"[ERROR] Failed: {pdb_code} rep {replica} | {e}")

pd.DataFrame(rmsd_rows).to_csv(os.path.join(out_dir, "rmsd_long.csv"), index=False)
pd.DataFrame(rmsf_rows).to_csv(os.path.join(out_dir, "rmsf_long.csv"), index=False)
pd.DataFrame(rmsf_pep_rows).to_csv(os.path.join(out_dir, "rmsf_pep_long.csv"), index=False)

print("\n[INFO] Metrics saved successfully.")

In [ ]:
eda_dir = "Analysis_results/eda"
os.makedirs(eda_dir, exist_ok=True)
df_all = pd.concat([df_md, df_gen], ignore_index=True)
eda_results = {}
eda_rows = []
eda_selection_string = (
    "chain A and resnum 1 to 270 and name CA or "
    "chain B and resnum 1 to 99 and name CA"
)
for i, row in df_all.iterrows():
    try:
        source = row.get("source")
        condition = row.get("condition", None)
        fraction = row.get("fraction", None)
        pdb_code = row["pdb_code"]
        replica = row["replica"]
        pdb_file = row["pdb_file"]
        dcd_path = row["dcd_path"]
        print(f"[INFO] EDA {i+1}/{len(df_all)} | {source} | {pdb_code} | rep {replica} | cond={condition} | frac={fraction}")
        syst = parsePDB(pdb_file)
        traj = parseDCD(dcd_path)
        traj.setAtoms(syst)
        sel = syst.select(eda_selection_string)
        if sel is None:
            print(f"[WARNING] Empty EDA selection: {source} | {pdb_code} | rep {replica} | cond={condition} | frac={fraction}")
            continue
        traj.setAtoms(sel)
        traj.superpose()
        eda = EDA()
        eda.buildCovariance(traj)
        eda.calcModes()
        key = (source, condition, fraction, pdb_code, replica)
        eda_results[key] = {
            "eda": eda,
            "selection": eda_selection_string,
            "n_atoms_selected": sel.numAtoms(),
            "n_modes": eda.numModes()
        }
        eda_rows.append({
            "source": source,
            "condition": condition,
            "fraction": fraction,
            "pdb_code": pdb_code,
            "replica": replica,
            "selection": eda_selection_string,
            "n_atoms_selected": sel.numAtoms(),
            "n_modes": eda.numModes(),
            "dcd_path": dcd_path,
            "pdb_file": pdb_file
        })
    except Exception as e:
        print(f"[ERROR] EDA failed: {row.get('source')} | {row.get('pdb_code')} | rep {row.get('replica')} | cond={row.get('condition', None)} | frac={row.get('fraction', None)} | {e}")

df_eda_metadata = pd.DataFrame(eda_rows)
df_eda_metadata.to_csv(os.path.join(eda_dir, "eda_metadata.csv"), index=False)
print("\n[INFO] EDA preparation completed.")
print(f"[INFO] Successful EDA objects: {len(df_eda_metadata)}")

In [38]:
subspace_rows = []
keys = list(eda_results.keys())
for i, key1 in enumerate(keys):
    eda1 = eda_results[key1]["eda"]
    for j, key2 in enumerate(keys):
        eda2 = eda_results[key2]["eda"]
        try:
            so = calcSubspaceOverlap(eda1, eda2)
            subspace_rows.append({
                "source1": key1[0],
                "condition1": key1[1],
                "fraction1": key1[2],
                "pdb_code1": key1[3],
                "replica1": key1[4],
                "source2": key2[0],
                "condition2": key2[1],
                "fraction2": key2[2],
                "pdb_code2": key2[3],
                "replica2": key2[4],
                "subspace_overlap": float(so)
            })
        except Exception as e:
            print(
                f"[ERROR] Subspace overlap failed: "
                f"{key1} vs {key2} | {e}"
            )
df_subspace_overlap = pd.DataFrame(subspace_rows)
df_subspace_overlap.to_csv(
    os.path.join(eda_dir, "subspace_overlap_long.csv"),
    index=False
)
print("\n[INFO] Subspace overlap calculation completed.")
print(f"[INFO] Total comparisons: {len(df_subspace_overlap)}")


[INFO] Subspace overlap calculation completed.
[INFO] Total comparisons: 121104


In [39]:
df_subspace_overlap["replica1"] = df_subspace_overlap["replica1"].astype(str)
df_subspace_overlap["replica2"] = df_subspace_overlap["replica2"].astype(str)

df_subspace_overlap["label1"] = (
    df_subspace_overlap["source1"].astype(str) + "_" +
    df_subspace_overlap["pdb_code1"].astype(str) + "_rep" +
    df_subspace_overlap["replica1"].astype(str) + "_" +
    df_subspace_overlap["condition1"].astype(str) + "_f" +
    df_subspace_overlap["fraction1"].astype(str)
)

df_subspace_overlap["label2"] = (
    df_subspace_overlap["source2"].astype(str) + "_" +
    df_subspace_overlap["pdb_code2"].astype(str) + "_rep" +
    df_subspace_overlap["replica2"].astype(str) + "_" +
    df_subspace_overlap["condition2"].astype(str) + "_f" +
    df_subspace_overlap["fraction2"].astype(str)
)

df_subspace_overlap.to_csv(
    os.path.join(eda_dir, "subspace_overlap_long.csv"),
    index=False
)

In [40]:
matplotlib.rcParams['figure.constrained_layout.use'] = True
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.sans-serif'] = ['DejaVu Sans']
matplotlib.rcParams['font.size'] = 12
matplotlib.rcParams['lines.linewidth'] = 2

df_rmsd = pd.read_csv("Analysis_results/trajectory_metrics/rmsd_long.csv")

protein_label_map = {
    "pep_free": "No peptide",
    "3UTQ": "ALWGPDPAAA",
    "5C0F": "RQWGPDPAAV",
    "5N1Y": "MVWGPDPLYV"
}
protein_order = ["3UTQ", "5C0F", "5N1Y", "pep_free"]

base_fig_dir = "Analysis_results/figures/rmsd"
md_fig_dir = os.path.join(base_fig_dir, "md")
gen_fig_dir = os.path.join(base_fig_dir, "generated")
os.makedirs(md_fig_dir, exist_ok=True)
os.makedirs(gen_fig_dir, exist_ok=True)

def plot_mean_std(df, col, ax, color, label, shade=True):
    mean_values = df.groupby('frame')[col].mean()
    mean_values = mean_values.reindex(index=natsort.natsorted(mean_values.index))
    std_values = df.groupby('frame')[col].std()
    std_values = std_values.reindex(index=natsort.natsorted(std_values.index))
    if shade:
        ax.fill_between(
            mean_values.index,
            mean_values - std_values,
            mean_values + std_values,
            color=color,
            alpha=0.3
        )
    ax.plot(mean_values.index, mean_values, color=color, label=label)
    ax.legend(loc='upper left', frameon=False, handlelength=0)

def make_four_panel_rmsd_figure(df_subset, title=None):
    fig, axes = plt.subplots(2, 2, figsize=(12, 6), sharex=True, sharey=True)
    axes = axes.flatten()
    color_map = {
        "pep_free": "black",
        "3UTQ": "blue",
        "5C0F": "red",
        "5N1Y": "orange"
    }
    for ax, pdb_code in zip(axes, protein_order):
        df_p = df_subset[df_subset["pdb_code"] == pdb_code]
        if df_p.empty:
            ax.set_title(f"{pdb_code} (missing)")
            continue
        plot_mean_std(
            df=df_p,
            col="rmsd",
            ax=ax,
            color=color_map[pdb_code],
            label=protein_label_map[pdb_code]
        )
        ax.set_xlabel("Frame")
        ax.set_ylabel("RMSD (Å)")
    if title is not None:
        fig.suptitle(title)
    return fig

# MD RMSD figure
df_md_rmsd = df_rmsd[df_rmsd["source"] == "md"].copy()
fig = make_four_panel_rmsd_figure(
    df_md_rmsd,
    title="MD RMSD"
)
md_out = os.path.join(md_fig_dir, "rmsd_md_all_proteins.png")
fig.savefig(md_out, dpi=300, bbox_inches="tight")
plt.close(fig)
print(f"[INFO] Saved: {md_out}")

# Generated RMSD figures by condition and fraction

df_gen_rmsd = df_rmsd[df_rmsd["source"] == "generated"].copy()
conditions = sorted(df_gen_rmsd["condition"].dropna().unique())
fractions = sorted(df_gen_rmsd["fraction"].dropna().unique())
for condition in conditions:
    for fraction in fractions:
        df_cf = df_gen_rmsd[
            (df_gen_rmsd["condition"] == condition) &
            (df_gen_rmsd["fraction"] == fraction)
        ].copy()

        if df_cf.empty:
            print(f"[WARNING] Empty RMSD subset: {condition} | fraction {fraction}")
            continue
        out_dir = os.path.join(gen_fig_dir, condition, f"fraction_{int(fraction)}")
        os.makedirs(out_dir, exist_ok=True)

        fig = make_four_panel_rmsd_figure(
            df_cf,
            title=f"Generated RMSD | {condition} | fraction {int(fraction)}"
        )
        out_path = os.path.join(
            out_dir,
            f"rmsd_{condition}_f{int(fraction)}_all_proteins.png"
        )
        fig.savefig(out_path, dpi=300, bbox_inches="tight")
        plt.close(fig)

        print(f"[INFO] Saved: {out_path}")

[INFO] Saved: Analysis_results/figures/rmsd/md/rmsd_md_all_proteins.png
[INFO] Saved: Analysis_results/figures/rmsd/generated/not_shuffle_allframes/fraction_10/rmsd_not_shuffle_allframes_f10_all_proteins.png
[INFO] Saved: Analysis_results/figures/rmsd/generated/not_shuffle_allframes/fraction_20/rmsd_not_shuffle_allframes_f20_all_proteins.png
[INFO] Saved: Analysis_results/figures/rmsd/generated/not_shuffle_allframes/fraction_30/rmsd_not_shuffle_allframes_f30_all_proteins.png
[INFO] Saved: Analysis_results/figures/rmsd/generated/not_shuffle_allframes/fraction_40/rmsd_not_shuffle_allframes_f40_all_proteins.png
[INFO] Saved: Analysis_results/figures/rmsd/generated/not_shuffle_allframes/fraction_50/rmsd_not_shuffle_allframes_f50_all_proteins.png
[INFO] Saved: Analysis_results/figures/rmsd/generated/not_shuffle_allframes/fraction_60/rmsd_not_shuffle_allframes_f60_all_proteins.png
[INFO] Saved: Analysis_results/figures/rmsd/generated/not_shuffle_allframes/fraction_70/rmsd_not_shuffle_allfram

In [41]:
matplotlib.rcParams['figure.constrained_layout.use'] = True
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.sans-serif'] = ['DejaVu Sans']
matplotlib.rcParams['font.size'] = 12
matplotlib.rcParams['lines.linewidth'] = 2

df_rmsf = pd.read_csv("Analysis_results/trajectory_metrics/rmsf_long.csv")
df_rmsf_pep = pd.read_csv("Analysis_results/trajectory_metrics/rmsf_pep_long.csv")
protein_label_map = {
    "pep_free": "No peptide",
    "3UTQ": "ALWGPDPAAA",
    "5C0F": "RQWGPDPAAV",
    "5N1Y": "MVWGPDPLYV"
}
protein_order = ["3UTQ", "5C0F", "5N1Y", "pep_free"]
peptide_protein_order = ["3UTQ", "5C0F", "5N1Y"]
color_map = {
    "pep_free": "black",
    "3UTQ": "blue",
    "5C0F": "red",
    "5N1Y": "orange"
}

rmsf_fig_dir = "Analysis_results/figures/rmsf"
rmsf_pep_fig_dir = "Analysis_results/figures/rmsf_pep"
os.makedirs(rmsf_fig_dir, exist_ok=True)
os.makedirs(rmsf_pep_fig_dir, exist_ok=True)
def plot_mean_std(df, xcol, ycol, ax, color, label, shade=True):
    mean_values = df.groupby(xcol)[ycol].mean()
    mean_values = mean_values.reindex(index=natsort.natsorted(mean_values.index))

    std_values = df.groupby(xcol)[ycol].std()
    std_values = std_values.reindex(index=natsort.natsorted(std_values.index))

    if shade:
        ax.fill_between(
            mean_values.index,
            mean_values - std_values,
            mean_values + std_values,
            color=color,
            alpha=0.3
        )

    ax.plot(mean_values.index, mean_values, color=color, label=label)

def make_pep_rmsf_figure(df_subset, title=None):
    fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=True)
    axes = axes.flatten()

    for ax, pdb_code in zip(axes, peptide_protein_order):
        df_p = df_subset[df_subset["pdb_code"] == pdb_code]

        if df_p.empty:
            ax.set_title(f"{pdb_code} (missing)")
            continue
        df_p = df_p.copy()
        df_p["pep_num"] = df_p["pep_pos"].str.extract(r"(\d+)").astype(int)
        mean_values = df_p.groupby("pep_num")["rmsf_pep"].mean()
        std_values = df_p.groupby("pep_num")["rmsf_pep"].std()
        ax.fill_between(
            mean_values.index,
            mean_values - std_values,
            mean_values + std_values,
            color=color_map[pdb_code],
            alpha=0.3
        )
        ax.plot(mean_values.index, mean_values, color=color_map[pdb_code], label=protein_label_map[pdb_code])
        ax.set_xticks(range(1, 11))
        ax.set_xticklabels([f"P{i}" for i in range(1, 11)], rotation=45)
        ax.set_xlabel("Peptide position")
        ax.set_ylabel("RMSF")
        ax.legend(loc="best", frameon=False)
    if title is not None:
        fig.suptitle(title)
    return fig

def make_rmsf_figure(df_subset, title=None):
    fig, axes = plt.subplots(4, 1, figsize=(12, 8), sharex=True, sharey=True)
    axes = axes.flatten()
    for ax, pdb_code in zip(axes, protein_order):
        df_p = df_subset[df_subset["pdb_code"] == pdb_code]
        if df_p.empty:
            ax.set_title(f"{pdb_code} (missing)")
            continue
        plot_mean_std(
            df=df_p,
            xcol="resindex",
            ycol="rmsf",
            ax=ax,
            color=color_map[pdb_code],
            label=protein_label_map[pdb_code]
        )
        ax.axvline(180, linestyle="--", linewidth=1, color="black")
        ax.axvline(274, linestyle="--", linewidth=1, color="black")
        ax.set_ylim(0, 6)
        ax.grid(True, alpha=0.3)
        ax.legend(loc="best", frameon=False)
        ax.set_ylabel("RMSF")
    axes[-1].set_xlabel("Residue index")
    if title is not None:
        fig.suptitle(title)
    return fig

# MD figures

df_md_rmsf = df_rmsf[df_rmsf["source"] == "md"].copy()
df_md_rmsf_pep = df_rmsf_pep[df_rmsf_pep["source"] == "md"].copy()
md_rmsf_dir = os.path.join(rmsf_fig_dir, "md")
md_rmsf_pep_dir = os.path.join(rmsf_pep_fig_dir, "md")
os.makedirs(md_rmsf_dir, exist_ok=True)
os.makedirs(md_rmsf_pep_dir, exist_ok=True)
fig = make_rmsf_figure(df_md_rmsf, title="MD RMSF")
out_path = os.path.join(md_rmsf_dir, "rmsf_md_all_proteins.png")
fig.savefig(out_path, dpi=300, bbox_inches="tight")
plt.close(fig)
print(f"[INFO] Saved: {out_path}")
fig = make_pep_rmsf_figure(df_md_rmsf_pep, title="MD Peptide RMSF")
out_path = os.path.join(md_rmsf_pep_dir, "rmsf_pep_md_all_proteins.png")
fig.savefig(out_path, dpi=300, bbox_inches="tight")
plt.close(fig)
print(f"[INFO] Saved: {out_path}")

# Generated figures by condition + fraction
df_gen_rmsf = df_rmsf[df_rmsf["source"] == "generated"].copy()
df_gen_rmsf_pep = df_rmsf_pep[df_rmsf_pep["source"] == "generated"].copy()
conditions = sorted(df_gen_rmsf["condition"].dropna().unique())
fractions = sorted(df_gen_rmsf["fraction"].dropna().unique())
for condition in conditions:
    for fraction in fractions:
        # ---- RMSF
        df_cf_rmsf = df_gen_rmsf[
            (df_gen_rmsf["condition"] == condition) &
            (df_gen_rmsf["fraction"] == fraction)
        ].copy()
        if not df_cf_rmsf.empty:
            out_dir = os.path.join(rmsf_fig_dir, "generated", condition, f"fraction_{int(fraction)}")
            os.makedirs(out_dir, exist_ok=True)
            fig = make_rmsf_figure(
                df_cf_rmsf,
                title=f"Generated RMSF | {condition} | fraction {int(fraction)}"
            )
            out_path = os.path.join(
                out_dir,
                f"rmsf_{condition}_f{int(fraction)}_all_proteins.png"
            )
            fig.savefig(out_path, dpi=300, bbox_inches="tight")
            plt.close(fig)
            print(f"[INFO] Saved: {out_path}")
        df_cf_rmsf_pep = df_gen_rmsf_pep[
            (df_gen_rmsf_pep["condition"] == condition) &
            (df_gen_rmsf_pep["fraction"] == fraction)
        ].copy()
        if not df_cf_rmsf_pep.empty:
            out_dir = os.path.join(rmsf_pep_fig_dir, "generated", condition, f"fraction_{int(fraction)}")
            os.makedirs(out_dir, exist_ok=True)
            fig = make_pep_rmsf_figure(
                df_cf_rmsf_pep,
                title=f"Generated Peptide RMSF | {condition} | fraction {int(fraction)}"
            )
            out_path = os.path.join(
                out_dir,
                f"rmsf_pep_{condition}_f{int(fraction)}_all_proteins.png"
            )
            fig.savefig(out_path, dpi=300, bbox_inches="tight")
            plt.close(fig)
            print(f"[INFO] Saved: {out_path}")

[INFO] Saved: Analysis_results/figures/rmsf/md/rmsf_md_all_proteins.png
[INFO] Saved: Analysis_results/figures/rmsf_pep/md/rmsf_pep_md_all_proteins.png
[INFO] Saved: Analysis_results/figures/rmsf/generated/not_shuffle_allframes/fraction_10/rmsf_not_shuffle_allframes_f10_all_proteins.png
[INFO] Saved: Analysis_results/figures/rmsf_pep/generated/not_shuffle_allframes/fraction_10/rmsf_pep_not_shuffle_allframes_f10_all_proteins.png
[INFO] Saved: Analysis_results/figures/rmsf/generated/not_shuffle_allframes/fraction_20/rmsf_not_shuffle_allframes_f20_all_proteins.png
[INFO] Saved: Analysis_results/figures/rmsf_pep/generated/not_shuffle_allframes/fraction_20/rmsf_pep_not_shuffle_allframes_f20_all_proteins.png
[INFO] Saved: Analysis_results/figures/rmsf/generated/not_shuffle_allframes/fraction_30/rmsf_not_shuffle_allframes_f30_all_proteins.png
[INFO] Saved: Analysis_results/figures/rmsf_pep/generated/not_shuffle_allframes/fraction_30/rmsf_pep_not_shuffle_allframes_f30_all_proteins.png
[INFO] S

In [42]:
df = pd.read_csv("Analysis_results/eda/subspace_overlap_long.csv")  
value_col = "subspace_overlap"
out_dir = "Analysis_results/figures/rmsip_heatmaps"
os.makedirs(out_dir, exist_ok=True)
protein_order = ["3UTQ", "5C0F", "5N1Y", "pep_free"]
replica_order = [0, 1, 2]
fraction_order = [10, 20, 30, 40, 50, 60, 70]
protein_rank = {p: i for i, p in enumerate(protein_order)}
replica_rank = {r: i for i, r in enumerate(replica_order)}
fraction_rank = {f: i for i, f in enumerate(fraction_order)}
condition_order = [
    "shuffle_allframes",
    "shuffle_not_allframes",
    "not_shuffle_allframes",
    "not_shuffle_not_allframes"
]
condition_rank = {c: i for i, c in enumerate(condition_order)}

# Label
def short_label(row, suffix):
    return f"{row[f'pdb_code{suffix}']}_rep_{row[f'replica{suffix}']}_frac{int(row[f'fraction{suffix}'])}"
def full_label(row, suffix):
    return f"{row[f'condition{suffix}']}__{row[f'pdb_code{suffix}']}_rep_{row[f'replica{suffix}']}_frac{int(row[f'fraction{suffix}'])}"
def md_label(row, suffix):
    return f"{row[f'pdb_code{suffix}']}_rep_{row[f'replica{suffix}']}"
def make_generated_label_list(include_condition=False):
    labels = []
    if include_condition:
        for condition in condition_order:
            for pdb_code in protein_order:
                for replica in replica_order:
                    for fraction in fraction_order:
                        labels.append(f"{condition}__{pdb_code}_rep_{replica}_frac{fraction}")
    else:
        for pdb_code in protein_order:
            for replica in replica_order:
                for fraction in fraction_order:
                    labels.append(f"{pdb_code}_rep_{replica}_frac{fraction}")
    return labels
def make_md_label_list():
    labels = []
    for pdb_code in protein_order:
        for replica in replica_order:
            labels.append(f"{pdb_code}_rep_{replica}")
    return labels

# MD vs MD heatmap
df_md = df[
    (df["source1"] == "md") &
    (df["source2"] == "md")
].copy()
df_md["label1_md"] = df_md.apply(lambda x: md_label(x, "1"), axis=1)
df_md["label2_md"] = df_md.apply(lambda x: md_label(x, "2"), axis=1)
pivot_md = df_md.pivot(
    index="label1_md",
    columns="label2_md",
    values=value_col
)
ordered_md_labels = make_md_label_list()
pivot_md = pivot_md.reindex(index=ordered_md_labels, columns=ordered_md_labels)
plt.figure(figsize=(8, 7))
sns.heatmap(
    pivot_md,
    cmap="YlGnBu",
    square=True
)
plt.title(f"MD vs MD | {value_col} heatmap")
plt.xlabel("")
plt.ylabel("")
plt.xticks(rotation=90, fontsize=8)
plt.yticks(rotation=0, fontsize=8)
out_path = os.path.join(out_dir, f"heatmap_md_vs_md_{value_col}.png")
plt.savefig(out_path, dpi=300, bbox_inches="tight")
plt.close()
print(f"[INFO] Saved: {out_path}")

# Generated vs Generated, one heatmap per condition (84x84)
df_gen = df[
    (df["source1"] == "generated") &
    (df["source2"] == "generated")
].copy()
df_gen["label1_short"] = df_gen.apply(lambda x: short_label(x, "1"), axis=1)
df_gen["label2_short"] = df_gen.apply(lambda x: short_label(x, "2"), axis=1)
df_gen["label1_full"] = df_gen.apply(lambda x: full_label(x, "1"), axis=1)
df_gen["label2_full"] = df_gen.apply(lambda x: full_label(x, "2"), axis=1)
for condition in condition_order:
    df_cond = df_gen[
        (df_gen["condition1"] == condition) &
        (df_gen["condition2"] == condition)
    ].copy()
    if df_cond.empty:
        print(f"[WARNING] Empty condition subset: {condition}")
        continue
    pivot_cond = df_cond.pivot(
        index="label1_short",
        columns="label2_short",
        values=value_col
    )
    ordered_cond_labels = make_generated_label_list(include_condition=False)
    pivot_cond = pivot_cond.reindex(index=ordered_cond_labels, columns=ordered_cond_labels)
    plt.figure(figsize=(20, 18))
    sns.heatmap(
        pivot_cond,
        cmap="YlGnBu",
        square=True
    )
    plt.title(f"{condition} | {value_col} heatmap")
    plt.xlabel("")
    plt.ylabel("")
    plt.xticks(rotation=90, fontsize=7)
    plt.yticks(rotation=0, fontsize=7)
    out_path = os.path.join(out_dir, f"heatmap_{condition}_{value_col}.png")
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"[INFO] Saved: {out_path}")

# All generated results (336x336)
pivot_all = df_gen.pivot(
    index="label1_full",
    columns="label2_full",
    values=value_col
)
ordered_all_labels = make_generated_label_list(include_condition=True)
pivot_all = pivot_all.reindex(index=ordered_all_labels, columns=ordered_all_labels)
plt.figure(figsize=(42, 36))
sns.heatmap(
    pivot_all,
    cmap="YlGnBu",
    square=True
)
plt.title(f"All generated results | {value_col} heatmap")
plt.xlabel("")
plt.ylabel("")
plt.xticks(rotation=90, fontsize=4)
plt.yticks(rotation=0, fontsize=4)
out_path = os.path.join(out_dir, f"heatmap_all_generated_{value_col}.png")
plt.savefig(out_path, dpi=300, bbox_inches="tight")
plt.close()
print(f"[INFO] Saved: {out_path}")

[INFO] Saved: Analysis_results/figures/rmsip_heatmaps/heatmap_md_vs_md_subspace_overlap.png
[INFO] Saved: Analysis_results/figures/rmsip_heatmaps/heatmap_shuffle_allframes_subspace_overlap.png
[INFO] Saved: Analysis_results/figures/rmsip_heatmaps/heatmap_shuffle_not_allframes_subspace_overlap.png
[INFO] Saved: Analysis_results/figures/rmsip_heatmaps/heatmap_not_shuffle_allframes_subspace_overlap.png
[INFO] Saved: Analysis_results/figures/rmsip_heatmaps/heatmap_not_shuffle_not_allframes_subspace_overlap.png
[INFO] Saved: Analysis_results/figures/rmsip_heatmaps/heatmap_all_generated_subspace_overlap.png


In [ ]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


df = pd.read_csv("Analysis_results/eda/subspace_overlap_long.csv")

# Metric column

value_col = "subspace_overlap"



# Keep only generated vs md
df_gm = df[
    (df["source1"] == "generated") &
    (df["source2"] == "md")
].copy()


protein_order = ["3UTQ", "5C0F", "5N1Y", "pep_free"]
replica_order = [0, 1, 2]
fraction_order = [10, 20, 30, 40, 50, 60, 70]
condition_order = [
    "shuffle_allframes",
    "shuffle_not_allframes",
    "not_shuffle_allframes",
    "not_shuffle_not_allframes"
]

condition_map = {
    "shuffle_allframes": "C1",
    "shuffle_not_allframes": "C2",
    "not_shuffle_allframes": "C3",
    "not_shuffle_not_allframes": "C4"
}

def md_label(row):
    return f"{row['pdb_code2']}_rep{int(row['replica2'])}"

def gen_label(row):
    cond = condition_map[row["condition1"]]
    return f"{row['pdb_code1']}_{cond}_R{int(row['replica1'])}_F{int(row['fraction1'])}"

df_gm["md_label"] = df_gm.apply(md_label, axis=1)
df_gm["gen_label"] = df_gm.apply(gen_label, axis=1)

# Expected label orders
ordered_md_labels = [
    f"{p}_rep{r}"
    for p in protein_order
    for r in replica_order
]

ordered_gen_labels = [
    f"{p}_{condition_map[c]}_R{r}_F{f}"
    for c in condition_order
    for p in protein_order
    for r in replica_order
    for f in fraction_order
]
# Build 12 x 336 matrix
# rows = MD, cols = generated

pivot_md_gen = df_gm.pivot(
    index="md_label",
    columns="gen_label",
    values=value_col
)

pivot_md_gen = pivot_md_gen.reindex(
    index=ordered_md_labels,
    columns=ordered_gen_labels
)


out_dir = "Analysis_results/figures/rmsip_heatmaps"
os.makedirs(out_dir, exist_ok=True)


plt.figure(figsize=(48, 8))
sns.heatmap(
    pivot_md_gen,
    cmap="YlGnBu"
)
plt.title(f"MD vs Generated | {value_col}")
plt.xlabel("Generated systems")
plt.ylabel("MD reference trajectories")
plt.xticks(rotation=90, fontsize=5)
plt.yticks(rotation=0, fontsize=8)
legend_text = (
    "C1 = shuffle_allframes\n"
    "C2 = shuffle_not_allframes\n"
    "C3 = not_shuffle_allframes\n"
    "C4 = not_shuffle_not_allframes\n"
    "R = replica\n"
    "F = fraction"
)

plt.gcf().text(
    1.01, 0.5, legend_text,
    fontsize=10,
    va="center"
)
out_path = os.path.join(out_dir, f"heatmap_md_vs_generated_{value_col}.png")
plt.savefig(out_path, dpi=300, bbox_inches="tight")
plt.close()

print(f"[INFO] Saved: {out_path}")



[INFO] Saved: Analysis_results/figures/rmsip_heatmaps/heatmap_md_vs_generated_subspace_overlap.png


In [ ]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


df = pd.read_csv("Analysis_results/eda/subspace_overlap_long.csv")  

# Metric column
if "rmsip" in df.columns:
    value_col = "rmsip"
elif "subspace_overlap" in df.columns:
    value_col = "subspace_overlap"
else:
    raise ValueError("Neither 'rmsip' nor 'subspace_overlap' column was found.")

# Keep only generated vs generated
df_gen = df[
    (df["source1"] == "generated") &
    (df["source2"] == "generated")
].copy()


protein_order = ["3UTQ", "5C0F", "5N1Y", "pep_free"]
replica_order = [0, 1, 2]
fraction_order = [10, 20, 30, 40, 50, 60, 70]
condition_order = [
    "shuffle_allframes",
    "shuffle_not_allframes",
    "not_shuffle_allframes",
    "not_shuffle_not_allframes"
]


def simple_label(row, suffix):
    return f"{row[f'pdb_code{suffix}']}_rep{int(row[f'replica{suffix}'])}"

df_gen["label1_simple"] = df_gen.apply(lambda x: simple_label(x, "1"), axis=1)
df_gen["label2_simple"] = df_gen.apply(lambda x: simple_label(x, "2"), axis=1)

ordered_simple_labels = [
    f"{p}_rep{r}"
    for p in protein_order
    for r in replica_order
]


out_dir = "Analysis_results/figures/rmsip_heatmaps/all_systems"
os.makedirs(out_dir, exist_ok=True)

# One heatmap per condition + fraction
# Each one should be 12 x 12
for condition in condition_order:
    for fraction in fraction_order:
        df_cf = df_gen[
            (df_gen["condition1"] == condition) &
            (df_gen["condition2"] == condition) &
            (df_gen["fraction1"] == fraction) &
            (df_gen["fraction2"] == fraction)
        ].copy()

        if df_cf.empty:
            print(f"[WARNING] Empty subset: {condition} | fraction {fraction}")
            continue

        pivot_cf = df_cf.pivot(
            index="label1_simple",
            columns="label2_simple",
            values=value_col
        )

        pivot_cf = pivot_cf.reindex(
            index=ordered_simple_labels,
            columns=ordered_simple_labels
        )

        plt.figure(figsize=(8, 7))
        sns.heatmap(
            pivot_cf,
            cmap="YlGnBu",
            square=True,
            annot=False
        )
        plt.title(f"{condition}_frac{fraction}")
        plt.xlabel("")
        plt.ylabel("")
        plt.xticks(rotation=90, fontsize=8)
        plt.yticks(rotation=0, fontsize=8)

        out_path = os.path.join(
            out_dir,
            f"heatmap_{condition}_frac{fraction}_{value_col}.png"
        )
        plt.savefig(out_path, dpi=300, bbox_inches="tight")
        plt.close()

        print(f"[INFO] Saved: {out_path}")

[INFO] Saved: Analysis_results/figures/rmsip_heatmaps/all_systems/heatmap_shuffle_allframes_frac10_subspace_overlap.png
[INFO] Saved: Analysis_results/figures/rmsip_heatmaps/all_systems/heatmap_shuffle_allframes_frac20_subspace_overlap.png
[INFO] Saved: Analysis_results/figures/rmsip_heatmaps/all_systems/heatmap_shuffle_allframes_frac30_subspace_overlap.png
[INFO] Saved: Analysis_results/figures/rmsip_heatmaps/all_systems/heatmap_shuffle_allframes_frac40_subspace_overlap.png
[INFO] Saved: Analysis_results/figures/rmsip_heatmaps/all_systems/heatmap_shuffle_allframes_frac50_subspace_overlap.png
[INFO] Saved: Analysis_results/figures/rmsip_heatmaps/all_systems/heatmap_shuffle_allframes_frac60_subspace_overlap.png
[INFO] Saved: Analysis_results/figures/rmsip_heatmaps/all_systems/heatmap_shuffle_allframes_frac70_subspace_overlap.png
[INFO] Saved: Analysis_results/figures/rmsip_heatmaps/all_systems/heatmap_shuffle_not_allframes_frac10_subspace_overlap.png
[INFO] Saved: Analysis_results/figur

In [ ]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv("Analysis_results/eda/subspace_overlap_long.csv")

value_col = "subspace_overlap"   # eğer rmsip csv kullanıyorsan bunu "rmsip" yap

# Keep only generated vs md

df_gm = df[
    (df["source1"] == "generated") &
    (df["source2"] == "md")
].copy()


protein_order = ["3UTQ", "5C0F", "5N1Y", "pep_free"]
replica_order = [0, 1, 2]
fraction_order = [10, 20, 30, 40, 50, 60, 70]
condition_order = [
    "shuffle_allframes",
    "shuffle_not_allframes",
    "not_shuffle_allframes",
    "not_shuffle_not_allframes"
]

condition_map = {
    "shuffle_allframes": "C1",
    "shuffle_not_allframes": "C2",
    "not_shuffle_allframes": "C3",
    "not_shuffle_not_allframes": "C4"
}


def md_label(row):
    return f"{row['pdb_code2']}_rep{int(row['replica2'])}"

def gen_label(row):
    cond = condition_map[row["condition1"]]
    return f"{row['pdb_code1']}_{cond}_R{int(row['replica1'])}_F{int(row['fraction1'])}"

df_gm["md_label"] = df_gm.apply(md_label, axis=1)
df_gm["gen_label"] = df_gm.apply(gen_label, axis=1)

# Fixed MD ordering
ordered_md_labels = [
    f"{p}_rep{r}"
    for p in protein_order
    for r in replica_order
]

# Output dir

out_dir = "Analysis_results/figures/rmsip_heatmaps/all_system_md_vs_gen"
os.makedirs(out_dir, exist_ok=True)


# One 12x12 heatmap per condition + fraction
for condition in condition_order:
    for fraction in fraction_order:

        df_cf = df_gm[
            (df_gm["condition1"] == condition) &
            (df_gm["fraction1"] == fraction)
        ].copy()

        if df_cf.empty:
            print(f"[WARNING] Empty subset: {condition} | fraction {fraction}")
            continue

        # generated label order for this single condition+fraction
        ordered_gen_labels = [
            f"{p}_{condition_map[condition]}_R{r}_F{fraction}"
            for p in protein_order
            for r in replica_order
        ]

        pivot_cf = df_cf.pivot(
            index="md_label",
            columns="gen_label",
            values=value_col
        )

        pivot_cf = pivot_cf.reindex(
            index=ordered_md_labels,
            columns=ordered_gen_labels
        )

        plt.figure(figsize=(10, 8))
        sns.heatmap(
            pivot_cf,
            cmap="YlGnBu",
            square=True
        )
        plt.title(f"{condition} | F{fraction} | MD vs Generated")
        plt.xlabel("Generated systems")
        plt.ylabel("MD reference trajectories")
        plt.xticks(rotation=90, fontsize=8)
        plt.yticks(rotation=0, fontsize=8)

        legend_text = (
            "C1 = shuffle_allframes\n"
            "C2 = shuffle_not_allframes\n"
            "C3 = not_shuffle_allframes\n"
            "C4 = not_shuffle_not_allframes\n"
            "R = replica\n"
            "F = fraction"
        )

        plt.gcf().text(
            1.02, 0.5, legend_text,
            fontsize=9,
            va="center"
        )

        out_path = os.path.join(
            out_dir,
            f"heatmap_md_vs_gen_{condition}_F{fraction}_{value_col}.png"
        )
        plt.savefig(out_path, dpi=300, bbox_inches="tight")
        plt.close()

        print(f"[INFO] Saved: {out_path}")

[INFO] Saved: Analysis_results/figures/rmsip_heatmaps/all_system_md_vs_gen/heatmap_md_vs_gen_shuffle_allframes_F10_subspace_overlap.png
[INFO] Saved: Analysis_results/figures/rmsip_heatmaps/all_system_md_vs_gen/heatmap_md_vs_gen_shuffle_allframes_F20_subspace_overlap.png
[INFO] Saved: Analysis_results/figures/rmsip_heatmaps/all_system_md_vs_gen/heatmap_md_vs_gen_shuffle_allframes_F30_subspace_overlap.png
[INFO] Saved: Analysis_results/figures/rmsip_heatmaps/all_system_md_vs_gen/heatmap_md_vs_gen_shuffle_allframes_F40_subspace_overlap.png
[INFO] Saved: Analysis_results/figures/rmsip_heatmaps/all_system_md_vs_gen/heatmap_md_vs_gen_shuffle_allframes_F50_subspace_overlap.png
[INFO] Saved: Analysis_results/figures/rmsip_heatmaps/all_system_md_vs_gen/heatmap_md_vs_gen_shuffle_allframes_F60_subspace_overlap.png
[INFO] Saved: Analysis_results/figures/rmsip_heatmaps/all_system_md_vs_gen/heatmap_md_vs_gen_shuffle_allframes_F70_subspace_overlap.png
[INFO] Saved: Analysis_results/figures/rmsip_hea

In [ ]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


df = pd.read_csv("Analysis_results/eda/subspace_overlap_long.csv")

value_col = "subspace_overlap"   

# Keep only generated vs md
df_gm = df[
    (df["source1"] == "generated") &
    (df["source2"] == "md")
].copy()

# Only matched protein + replica pairs


df_diag = df_gm[
    (df_gm["pdb_code1"] == df_gm["pdb_code2"]) &
    (df_gm["replica1"] == df_gm["replica2"])
].copy()


protein_order = ["3UTQ", "5C0F", "5N1Y", "pep_free"]
replica_order = [0, 1, 2]
fraction_order = [10, 20, 30, 40, 50, 60, 70]
condition_order = [
    "shuffle_allframes",
    "shuffle_not_allframes",
    "not_shuffle_allframes",
    "not_shuffle_not_allframes"
]

condition_map = {
    "shuffle_allframes": "C1",
    "shuffle_not_allframes": "C2",
    "not_shuffle_allframes": "C3",
    "not_shuffle_not_allframes": "C4"
}


df_diag["md_label"] = df_diag.apply(
    lambda x: f"{x['pdb_code2']}_rep{int(x['replica2'])}",
    axis=1
)

df_diag["system_label"] = df_diag.apply(
    lambda x: f"{condition_map[x['condition1']]}_F{int(x['fraction1'])}",
    axis=1
)

ordered_md_labels = [
    f"{p}_rep{r}"
    for p in protein_order
    for r in replica_order
]

ordered_system_labels = [
    f"{condition_map[c]}_F{f}"
    for c in condition_order
    for f in fraction_order
]

# Build 12 x 28 matrix
# rows = matched MD trajectories
# cols = systems (condition + fraction)
pivot_diag = df_diag.pivot(
    index="md_label",
    columns="system_label",
    values=value_col
)

pivot_diag = pivot_diag.reindex(
    index=ordered_md_labels,
    columns=ordered_system_labels
)


out_dir = "Analysis_results/figures/rmsip_heatmaps"
os.makedirs(out_dir, exist_ok=True)

#
plt.figure(figsize=(18, 8))
sns.heatmap(
    pivot_diag,
    cmap="YlGnBu",
    annot=False
)
plt.title(f"Matched MD vs Generated diagonal summary | {value_col}")
plt.xlabel("Systems")
plt.ylabel("MD reference trajectories")
plt.xticks(rotation=90, fontsize=9)
plt.yticks(rotation=0, fontsize=9)

legend_text = (
    "C1 = shuffle_allframes\n"
    "C2 = shuffle_not_allframes\n"
    "C3 = not_shuffle_allframes\n"
    "C4 = not_shuffle_not_allframes\n"
    "F = fraction"
)

plt.gcf().text(
    0.85, 0.5, legend_text,
    fontsize=14,
    va="center"
)

out_path = os.path.join(out_dir, f"heatmap_md_vs_generated_diagonal_summary_{value_col}.png")
plt.savefig(out_path, dpi=300, bbox_inches="tight")
plt.close()

print(f"[INFO] Saved: {out_path}")

[INFO] Saved: Analysis_results/figures/rmsip_heatmaps/heatmap_md_vs_generated_diagonal_summary_subspace_overlap.png


In [ ]:

# Violin plot for heatmap columns
# Each system column has 12 matched MD-vs-generated values

violin_df = df_diag.copy()


violin_df["system_label"] = violin_df.apply(
    lambda x: f"{condition_map[x['condition1']]}_F{int(x['fraction1'])}",
    axis=1
)

violin_df["md_label"] = violin_df.apply(
    lambda x: f"{x['pdb_code2']}_rep{int(x['replica2'])}",
    axis=1
)

# sıralama
violin_df["system_label"] = pd.Categorical(
    violin_df["system_label"],
    categories=ordered_system_labels,
    ordered=True
)

plt.figure(figsize=(18, 7))

sns.violinplot(
    data=violin_df,
    x="system_label",
    y=value_col,
    order=ordered_system_labels,
    inner=None,
    cut=0,
    color="lightgray"
)

sns.stripplot(
    data=violin_df,
    x="system_label",
    y=value_col,
    order=ordered_system_labels,
    color="black",
    size=4,
    alpha=0.7,
    jitter=True
)

plt.title(f"Matched MD vs Generated diagonal distribution | {value_col}")
plt.xlabel("Systems")
plt.ylabel(value_col)
plt.xticks(rotation=90, fontsize=9)
plt.yticks(fontsize=10)

legend_text = (
    "C1 = shuffle_allframes\n"
    "C2 = shuffle_not_allframes\n"
    "C3 = not_shuffle_allframes\n"
    "C4 = not_shuffle_not_allframes\n"
    "F = fraction\n"
)

plt.gcf().text(
    0.95, 0.5, legend_text,
    fontsize=11,
    va="center"
)

out_path = os.path.join(
    out_dir,
    f"violin_md_vs_generated_diagonal_summary_{value_col}.png"
)

plt.savefig(out_path, dpi=300, bbox_inches="tight")
plt.close()

print(f"[INFO] Saved: {out_path}")

[INFO] Saved: Analysis_results/figures/rmsip_heatmaps/violin_md_vs_generated_diagonal_summary_subspace_overlap.png


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# SETTINGS
# ============================================================
condition = "not_shuffle_allframes"
fraction = 40

base_out = f"Analysis_results/figures/md_vs_generated_overlay/{condition}/fraction_{fraction}"
os.makedirs(base_out, exist_ok=True)

df_rmsd = pd.read_csv("Analysis_results/trajectory_metrics/rmsd_long.csv")
df_rmsf = pd.read_csv("Analysis_results/trajectory_metrics/rmsf_long.csv")

protein_order = ["3UTQ", "5C0F", "5N1Y", "pep_free"]

protein_label_map = {
    "3UTQ": "ALWGPDPAAA",
    "5C0F": "RQWGPDPAAV",
    "5N1Y": "MVWGPDPLYV",
    "pep_free": "No peptide"
}

color_map = {
    "3UTQ": "blue",
    "5C0F": "red",
    "5N1Y": "orange",
    "pep_free": "green"
}


def mean_std_by_x(df, xcol, ycol):
    grouped = df.groupby(xcol)[ycol]
    mean = grouped.mean()
    std = grouped.std()
    return mean.index.values, mean.values, std.values


def plot_overlay_line(df_md, df_gen, xcol, ycol, protein, ylabel, title, out_path):
    fig, ax = plt.subplots(figsize=(10, 5))

    # MD
    x_md, mean_md, std_md = mean_std_by_x(df_md, xcol, ycol)
    ax.plot(x_md, mean_md, color="black", label="MD", linewidth=2)
    ax.fill_between(x_md, mean_md - std_md, mean_md + std_md, color="black", alpha=0.2)

    # Generated
    x_gen, mean_gen, std_gen = mean_std_by_x(df_gen, xcol, ycol)
    c = color_map[protein]
    ax.plot(x_gen, mean_gen, color=c, label="Generated", linewidth=2)
    ax.fill_between(x_gen, mean_gen - std_gen, mean_gen + std_gen, color=c, alpha=0.25)

    ax.set_title(title)
    ax.set_xlabel(xcol)
    ax.set_ylabel(ylabel)
    ax.legend(frameon=False)
    ax.grid(alpha=0.3)

    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

    print(f"[INFO] Saved: {out_path}")


def regression_scatter(x, y, xlabel, ylabel, title, out_path):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]

    slope, intercept = np.polyfit(x, y, 1)
    y_pred = slope * x + intercept

    corr = np.corrcoef(x, y)[0, 1]
    r2 = corr ** 2

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(x, y, alpha=0.75)
    ax.plot(x, y_pred, color="black", linewidth=2)

    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(f"{title}\nR = {corr:.3f}, R² = {r2:.3f}")
    ax.grid(alpha=0.3)

    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

    print(f"[INFO] Saved: {out_path}")



for protein in protein_order:

    label = protein_label_map[protein]

    
    # RMSF overlay
    #
    md_rmsf_p = df_rmsf[
        (df_rmsf["source"] == "md") &
        (df_rmsf["pdb_code"] == protein)
    ].copy()

    gen_rmsf_p = df_rmsf[
        (df_rmsf["source"] == "generated") &
        (df_rmsf["pdb_code"] == protein) &
        (df_rmsf["condition"] == condition) &
        (df_rmsf["fraction"] == fraction)
    ].copy()

    out_path = os.path.join(base_out, f"overlay_rmsf_md_vs_gen_{protein}_{condition}_F{fraction}.png")

    plot_overlay_line(
        df_md=md_rmsf_p,
        df_gen=gen_rmsf_p,
        xcol="resindex",
        ycol="rmsf",
        protein=protein,
        ylabel="RMSF (Å)",
        title=f"RMSF | MD vs Generated | {label} | {condition} | F{fraction}",
        out_path=out_path
    )

    # RMSF scatter/regression
    # residue-wise mean MD vs mean Generated
    md_rmsd_p = df_rmsd[
        (df_rmsd["source"] == "md") &
        (df_rmsd["pdb_code"] == protein)
    ].copy()

    gen_rmsd_p = df_rmsd[
        (df_rmsd["source"] == "generated") &
        (df_rmsd["pdb_code"] == protein) &
        (df_rmsd["condition"] == condition) &
        (df_rmsd["fraction"] == fraction)
    ].copy()

    md_rmsf_mean = md_rmsf_p.groupby("resindex")["rmsf"].mean()
    gen_rmsf_mean = gen_rmsf_p.groupby("resindex")["rmsf"].mean()

    common_res = md_rmsf_mean.index.intersection(gen_rmsf_mean.index)

    out_path = os.path.join(base_out, f"scatter_regression_rmsf_{protein}_{condition}_F{fraction}.png")

    regression_scatter(
        x=md_rmsf_mean.loc[common_res].values,
        y=gen_rmsf_mean.loc[common_res].values,
        xlabel="MD RMSF mean (Å)",
        ylabel="Generated RMSF mean (Å)",
        title=f"RMSF regression | {label} | {condition} | F{fraction}",
        out_path=out_path
    )

    
    # RMSD scatter/regression
    # distribution-based quantile comparison
    
    md_rmsd_vals = md_rmsd_p["rmsd"].dropna().values
    gen_rmsd_vals = gen_rmsd_p["rmsd"].dropna().values

    qs = np.linspace(0.01, 0.99, 99)
    md_q = np.quantile(md_rmsd_vals, qs)
    gen_q = np.quantile(gen_rmsd_vals, qs)

    out_path = os.path.join(base_out, f"scatter_regression_rmsd_quantile_{protein}_{condition}_F{fraction}.png")

    regression_scatter(
        x=md_q,
        y=gen_q,
        xlabel="MD RMSD quantiles (Å)",
        ylabel="Generated RMSD quantiles (Å)",
        title=f"RMSD quantile regression | {label} | {condition} | F{fraction}",
        out_path=out_path
    )

[INFO] Saved: Analysis_results/figures/md_vs_generated_overlay/not_shuffle_allframes/fraction_40/overlay_rmsf_md_vs_gen_3UTQ_not_shuffle_allframes_F40.png
[INFO] Saved: Analysis_results/figures/md_vs_generated_overlay/not_shuffle_allframes/fraction_40/scatter_regression_rmsf_3UTQ_not_shuffle_allframes_F40.png
[INFO] Saved: Analysis_results/figures/md_vs_generated_overlay/not_shuffle_allframes/fraction_40/scatter_regression_rmsd_quantile_3UTQ_not_shuffle_allframes_F40.png
[INFO] Saved: Analysis_results/figures/md_vs_generated_overlay/not_shuffle_allframes/fraction_40/overlay_rmsf_md_vs_gen_5C0F_not_shuffle_allframes_F40.png
[INFO] Saved: Analysis_results/figures/md_vs_generated_overlay/not_shuffle_allframes/fraction_40/scatter_regression_rmsf_5C0F_not_shuffle_allframes_F40.png
[INFO] Saved: Analysis_results/figures/md_vs_generated_overlay/not_shuffle_allframes/fraction_40/scatter_regression_rmsd_quantile_5C0F_not_shuffle_allframes_F40.png
[INFO] Saved: Analysis_results/figures/md_vs_gen